In [126]:
##Step 0: Install theses necesary packages
##Install google-api-python-client google-auth google-auth-oauthlib pandas beautifulsoup4 gspread
##install -q pillow

##Step 1: Import packages after installing
#Import for step 2
import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import gspread
#Import for step 3
import re, unicodedata
import pandas as pd
#Import for step 5
import re
from urllib.parse import urlparse, parse_qs
#Import for step 6
import re
from googleapiclient.http import MediaInMemoryUpload
#Import for step 7
import os
from bs4 import BeautifulSoup
#Import for step 8
import base64
import io
import requests
#Import for step 9
import unicodedata
#Import for step 10
from urllib.parse import urlparse
#Import for step 11
from PIL import Image, UnidentifiedImageError
#Import for step 14
from bs4 import BeautifulSoup
import re
#Import for step 17
from bs4 import BeautifulSoup, NavigableString
import html


In [148]:
##Step 2: Build the drive client
#Scopes: read-only for both Drive & Sheets
SCOPES = [
    "https://www.googleapis.com/auth/drive.file",
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
]
CLIENT_SECRETS = "client_secret.json"   # <-- must exist locally (Desktop OAuth)
TOKEN_PATH = "token.json"               # will be created on first run

def get_creds():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file("client_secret.json", SCOPES)
        creds = flow.run_local_server(port=0, prompt="consent")  # first run opens browser
        with open("token.json", "w") as f:
            f.write(creds.to_json())
    return creds
creds = get_creds()

# API clients
drive = build("drive", "v3", credentials=creds)
gc = gspread.authorize(creds)

print("Drive & Sheets clients ready")

Drive & Sheets clients ready


In [128]:
## Step 3: Read the sheet and return docs_url
SHEET_URL = "https://docs.google.com/spreadsheets/d/1sMgzVgxNzT5RcYHlrDlQkNKYkfwTWxiys1vYhNWF38Q/edit?gid=1887291779#gid=1887291779"
TAB_NAME  = "DSBV | Rackcosmo" 

ws = gc.open_by_url(SHEET_URL).worksheet(TAB_NAME)
rows = ws.get_all_values()
df = pd.DataFrame(rows[1:], columns=rows[0]) if rows else pd.DataFrame()
df.head()


##Step 3.1: Get the slug format of main keyword
def strip_diacritics(s: str) -> str:
    nfkd = unicodedata.normalize("NFD", s or "")
    return "".join(ch for ch in nfkd if unicodedata.category(ch) != "Mn")

def slugify(text: str) -> str:
    t = " ".join(str(text or "").strip().split())
    t = strip_diacritics(t).lower()
    t = re.sub(r"[^a-z0-9]+", "-", t)
    return re.sub(r"-{2,}", "-", t).strip("-")

def find_col(df_cols, candidates):
    norm = {re.sub(r"\s+", "", c).lower(): c for c in df_cols}
    for cand in candidates:
        k = re.sub(r"\s+", "", cand).lower()
        if k in norm:
            return norm[k]
    return None

if df.empty:
    raise RuntimeError("DataFrame `df` is empty. Run Step 3 first.")

# Locate the main keyword column (adjust candidates if your header differs)
mk_col = find_col(df.columns, ["main keyword", "Main Keyword", "main_keyword"])
if not mk_col:
    raise ValueError('Could not find a "main keyword" column in the sheet.')

# Add a slug column to df
df["main_keyword_slug"] = df[mk_col].map(slugify)

# Optional: build a separate list if you prefer not to rely on df later
main_keyword_slugs = [
    {
        "row_index": i + 2,  # head=1 => first data row is sheet row 2
        "main_keyword": str(val or "").strip(),
        "main_keyword_slug": slugify(val),
    }
    for i, val in enumerate(df[mk_col].tolist())
    if str(val or "").strip()
]

print(f"Created slugs for {len(main_keyword_slugs)} row(s).")
# Quick peek:
# df[["{mk_col}", "main_keyword_slug"]].head()
# main_keyword_slugs[:3]
df.head()

Created slugs for 34 row(s).


,STT,main keyword,Link docs Bài viết,Categories,Tags,Duyệt đăng,Trạng thái,Link nháp,main_keyword_slug
0,1,test Rackcosmo,https://docs.google.com/document/d/1tA_4WjRQ6m...,,,TRUE,Đã đăng tự động,https://rackcosmo.com/?p=1917,test-rackcosmo
1,2,kệ drive in,https://docs.google.com/document/d/1xP3fLHbpW5...,,,FALSE,,,ke-drive-in
2,3,kệ công nghiệp,https://docs.google.com/document/d/1I9cwdSrHb0...,,,FALSE,,,ke-cong-nghiep
3,4,kệ kho hàng,https://docs.google.com/document/d/1VrA5LaY7IE...,,,FALSE,,,ke-kho-hang
4,5,Kệ trung tải,https://docs.google.com/document/d/14lV24IGJAd...,,,FALSE,,,ke-trung-tai


In [129]:
## Step 4: Get docs_url_to_run

# === config ===
URL_COLUMN_NAME = "Link docs Bài viết"       # your URL column
SLUG_COLUMN_NAME = "main_keyword_slug"      # newly added slug column

# normalize headers
df.columns = [c.strip() for c in df.columns]

# sanity check
for col in ["Duyệt đăng", "Trạng thái", URL_COLUMN_NAME, SLUG_COLUMN_NAME]:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}. Found: {list(df.columns)}")

# helpers
def is_checked(v):
    s = "" if v is None else str(v).strip().lower()
    return s in {"true", "1", "y", "yes", "✓", "checked", "x"}

def is_blank(v):
    return (v is None) or (str(v).strip() == "")

# filter rows: "Duyệt đăng" ticked AND "Trạng thái" blank
mask = df["Duyệt đăng"].apply(is_checked) & df["Trạng thái"].apply(is_blank)

# keep URL + slug columns
filtered = df.loc[mask, [URL_COLUMN_NAME, SLUG_COLUMN_NAME]].copy()

# clean URLs and dedup by URL to keep alignment
filtered[URL_COLUMN_NAME] = filtered[URL_COLUMN_NAME].astype(str).str.strip().replace({"": pd.NA})
clean = (
    filtered
    .dropna(subset=[URL_COLUMN_NAME])
    .drop_duplicates(subset=[URL_COLUMN_NAME], keep="first")
)

# outputs (aligned 1:1)
docs_url_to_run = clean[URL_COLUMN_NAME].tolist()
main_keyword_slug_to_run = clean[SLUG_COLUMN_NAME].astype(str).str.strip().tolist()

print(f"docs_url_to_run → {len(docs_url_to_run)}")
print(f"main_keyword_slug_to_run → {len(main_keyword_slug_to_run)}")
# optional peek
# list(zip(docs_url_to_run[:5], main_keyword_slug_to_run[:5]))
print(docs_url_to_run[:5])
print(main_keyword_slug_to_run[:5])


docs_url_to_run → 1
main_keyword_slug_to_run → 1
['https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing']
['ke-trung-tai-4-tang']


In [130]:
## Step 5: Export docs to html
# Export Google Docs → HTML (in-memory, no saving)

try:
    drive  # use existing Drive client
except NameError:
    raise RuntimeError("Drive client `drive` not found. Run the auth cell first.")

# --- Helpers ---
def extract_file_id(url: str) -> str:
    """Extract a Drive fileId from common Docs/Drive URLs."""
    # /document/d/<ID>
    m = re.search(r"/document/d/([a-zA-Z0-9_-]+)", url)
    if m:
        return m.group(1)
    # /file/d/<ID>
    m = re.search(r"/file/d/([a-zA-Z0-9_-]+)", url)
    if m:
        return m.group(1)
    # ...?id=<ID>
    qs = parse_qs(urlparse(url).query)
    if "id" in qs and qs["id"]:
        return qs["id"][0]
    raise ValueError(f"Could not extract file ID from URL: {url}")

def export_doc_html_bytes(file_id: str) -> bytes:
    """Export a Google Doc to HTML bytes; raises if not a Doc."""
    meta = drive.files().get(fileId=file_id, fields="id,name,mimeType").execute()
    if meta["mimeType"] != "application/vnd.google-apps.document":
        raise TypeError(f"Not a Google Doc: {meta['name']} [{meta['mimeType']}]")
    return drive.files().export(fileId=file_id, mimeType="text/html").execute(), meta

# --- Main: export every URL in docs_url_to_run ---
try:
    docs_url_to_run  # list of URLs prepared earlier
except NameError:
    raise RuntimeError("`docs_url_to_run` is not defined. Build it from your sheet first.")

exported_html_docs = []  # list of dicts: {url, file_id, name, html} (html is a UTF-8 string)
exported_html_files = [] # List of html files after exported

for url in docs_url_to_run:
    try:
        fid = extract_file_id(url)
        html_bytes, meta = export_doc_html_bytes(fid)
        html_text = html_bytes.decode("utf-8", errors="ignore")   # <<< define html_text

        exported_html_docs.append({
            "url": url,
            "file_id": fid,
            "name": meta["name"],
            "html": html_text
        })
        exported_html_files.append(html_text)  # NEW: keep just the HTML string

        print(f"Exported: {meta['name']}")
    except TypeError as e:
        print(f"Skipping (not a Google Doc): {url} | {e}")
    except Exception as e:
        print(f"Failed: {url} | {e}")

print(f"\nDone. Exported {len(exported_html_docs)} Google Doc(s). "
      f"exported_html_files has {len(exported_html_files)} HTML strings.")
print(exported_html_docs)

Exported: Bài viết | 36 - kệ trung tải 4 tầng

Done. Exported 1 Google Doc(s). exported_html_files has 1 HTML strings.
[{'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing', 'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E', 'name': 'Bài viết | 36 - kệ trung tải 4 tầng', 'html': '<html><head><meta content="text/html; charset=UTF-8" http-equiv="content-type"><style type="text/css"> ul.lst-kix_sisju2akkx7v-7{list-style-type:none}.lst-kix_ksn62bjcvjdf-5 > li:before{content:"■  "}ul.lst-kix_sisju2akkx7v-8{list-style-type:none}ul.lst-kix_sisju2akkx7v-5{list-style-type:none}.lst-kix_ksn62bjcvjdf-4 > li:before{content:"○  "}.lst-kix_ksn62bjcvjdf-6 > li:before{content:"●  "}ul.lst-kix_sisju2akkx7v-6{list-style-type:none}ul.lst-kix_sisju2akkx7v-3{list-style-type:none}ul.lst-kix_sisju2akkx7v-4{list-style-type:none}ul.lst-kix_sisju2akkx7v-1{list-style-type:none}ul.lst-kix_sisju2akkx7v-2{list-style-type:none}ul.lst-kix_ipv0ujbgrru9-8{li

In [131]:
## Step 6: Upload exported HTML to Google Drive
try:
    drive  # Drive client from your auth step
    exported_html_docs  # list from Step 5
except NameError:
    raise RuntimeError("Missing `drive` or `exported_html_docs`. Run Step 5 first.")

# ====== REQUIRED: put your Drive folder ID here ======
DRIVE_FOLDER_ID = "1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX"
# =====================================================

def safe_filename(name: str) -> str:
    name = re.sub(r'[\\/:\*\?"<>\|]+', "_", name).strip()
    name = re.sub(r"[-\s]+", "-", name)
    return name or "exported_doc"

uploaded_html_files = []  # will hold dicts: {file_id, name, webViewLink, source_url, source_file_id}

for item in exported_html_docs:
    html_text = item["html"]
    base = safe_filename(item["name"])
    filename = f"{base}.html"

    media = MediaInMemoryUpload(
        html_text.encode("utf-8"),
        mimetype="text/html",
        resumable=False,
    )

    metadata = {
        "name": filename,
        "parents": [DRIVE_FOLDER_ID],
        "mimeType": "text/html",
    }

    try:
        file = (
            drive.files()
            .create(body=metadata, media_body=media, fields="id,name,webViewLink")
            # If uploading to a Shared Drive, uncomment the next line:
            # .create(body=metadata, media_body=media, fields="id,name,webViewLink", supportsAllDrives=True)
            .execute()
        )

        uploaded_html_files.append({
            "file_id": file["id"],
            "name": file["name"],
            "webViewLink": file.get("webViewLink"),
            "source_url": item["url"],
            "source_file_id": item["file_id"],
        })

        print(f"Uploaded: {file['name']}  →  {file['webViewLink']}")
    except Exception as e:
        print(f"Failed to upload {filename}: {e}")

print(f"\nDone. Uploaded {len(uploaded_html_files)} HTML file(s) to Drive folder {DRIVE_FOLDER_ID}.")


Uploaded: Bài-viết-_-36-kệ-trung-tải-4-tầng.html  →  https://drive.google.com/file/d/1RCgd0RtuKl6J_CL7J5ChBt7OVSAdricL/view?usp=drivesdk

Done. Uploaded 1 HTML file(s) to Drive folder 1NjOhY5tEOFUrz49YeQiA_fwGzKg_4CJX.


In [132]:
## Step 7: Process each local HTML file and extract <img> tags
from bs4 import BeautifulSoup

# Expect: exported_html_docs = [
#   {"url": ..., "file_id": ..., "name": ..., "html": "<!doctype html>..."},
#   ...
# ]

try:
    exported_html_docs  # list built in Step 5
except NameError:
    raise RuntimeError("`exported_html_docs` not found. Run the export step first.")

def extract_images_from_html_text(html_text: str) -> list[dict]:
    """Return a list of <img> tag attribute dicts from an HTML string."""
    soup = BeautifulSoup(html_text or "", "html.parser")
    return [dict(img.attrs) for img in soup.find_all("img")]

images_each_doc = []   # [{name, url, file_id, images:[{...}, ...]}, ...]
total_imgs = 0

for item in exported_html_docs:
    html_text = item.get("html", "") or ""
    imgs = extract_images_from_html_text(html_text)
    images_each_doc.append({
        "name": item.get("name"),
        "url": item.get("url"),
        "file_id": item.get("file_id"),
        "images": imgs,
    })
    total_imgs += len(imgs)

print(f"Extracted {total_imgs} <img> tag(s) across {len(images_each_doc)} HTML document(s).")
images_each_doc


Extracted 6 <img> tag(s) across 1 HTML document(s).


[{'name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXeg3YVeOKMv6LSZs-Sa5mGxNBN_NVvhE1HsYVQWJJi_nIRNu9iMlR02QhZIcWCKRs3BrpkWK1Xj2NGJu5hc-1yh0euLol4cD8Qaiz_L2g5eTCrqDbJHbC27gJzTSR_wiDWcTPObwA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': ''},
   {'alt': 'Bản vẽ chi tiết kệ',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXciuvN7msHn5wvUbaximAEko84LVKr2QXQaxUrVh7zIBoMDmGwXBm2i0Q42lT_MhEhVlwYdm_w0BWWB3iCKCy3UdgTz7ZxWXJTJh3SdkFGrl2EFgv3SfSf9PRRvEYW9ob5D4sX0BQ?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.6

In [133]:
## Step 8: Generate images slug and 5-word slug
try:
    images_each_doc  # from Step 9
except NameError:
    raise RuntimeError("`images_each_doc` not found. Run the image extraction step first.")

def strip_diacritics(s: str) -> str:
    # Remove accents (NFD -> drop combining marks)
    nfkd = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in nfkd if unicodedata.category(ch) != "Mn")

def slugify(text: str) -> str:
    if not text:
        return ""
    # normalize spaces, strip diacritics, to lower
    t = " ".join(str(text).strip().split())
    t = strip_diacritics(t).lower()
    # replace non-alphanum with hyphens
    t = re.sub(r"[^a-z0-9]+", "-", t)
    # collapse multiple hyphens and trim
    t = re.sub(r"-{2,}", "-", t).strip("-")
    return t

def first_n_words(text: str, n: int = 5) -> str:
    if not text:
        return ""
    words = str(text).strip().split()
    return " ".join(words[:n])

images_with_slugs = []  # [{doc_name, url, file_id, images:[{src, alt, slug_alt_full, slug_alt_first5, ...}]}]

for doc in images_each_doc:
    doc_entry = {
        "doc_name": doc.get("name"),
        "url": doc.get("url"),
        "file_id": doc.get("file_id"),
        "images": []
    }
    for attrs in doc.get("images", []):
        alt = attrs.get("alt", "") or ""
        slug_alt_full = slugify(alt)
        slug_alt_first5 = slugify(first_n_words(alt, 5))
        # copy existing attrs and add slugs
        enriched = dict(attrs)
        enriched["slug_alt_full"] = slug_alt_full
        enriched["slug_alt_first5"] = slug_alt_first5
        doc_entry["images"].append(enriched)
    images_with_slugs.append(doc_entry)

# Quick peek
total_imgs = sum(len(d["images"]) for d in images_with_slugs)
print(f"Added slugs for {total_imgs} image(s) across {len(images_with_slugs)} document(s).")
# Example:
# for d in images_with_slugs[:1]:
#     print(d["doc_name"], "→", len(d["images"]), "images")
#     for x in d["images"][:3]:
#         print(x.get("alt"), "=>", x["slug_alt_full"], "|", x["slug_alt_first5"])
images_with_slugs

Added slugs for 6 image(s) across 1 document(s).


[{'doc_name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXeg3YVeOKMv6LSZs-Sa5mGxNBN_NVvhE1HsYVQWJJi_nIRNu9iMlR02QhZIcWCKRs3BrpkWK1Xj2NGJu5hc-1yh0euLol4cD8Qaiz_L2g5eTCrqDbJHbC27gJzTSR_wiDWcTPObwA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'slug_alt_full': 'ke-trung-tai-4-tang',
    'slug_alt_first5': 'ke-trung-tai-4-tang'},
   {'alt': 'Bản vẽ chi tiết kệ',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXciuvN7msHn5wvUbaximAEko84LVKr2QXQaxUrVh7zIBoMDmGwXBm2i0Q42lT_MhEhVlwYdm_w0BWWB3iCKCy3UdgTz7ZxWXJTJh3SdkFGrl2EFgv3SfS

In [134]:
## Step 09: Download images using images_with_slugs
try:
    images_with_slugs  # [{doc_name, url, file_id, images:[{src, alt, slug_alt_full, slug_alt_first5, ...}]}]
except NameError:
    raise RuntimeError("`images_with_slugs` not found. Run the slug-building step first.")

RAW_DIR = "images_raw"
os.makedirs(RAW_DIR, exist_ok=True)

downloaded_image_paths: dict[str, str] = {}
download_errors: list[dict] = []

def safe_name(s: str) -> str:
    return re.sub(r'[\\/:*?"<>|]+', "_", (s or "").strip()) or "document"

def ensure_unique_path(base_dir: str, filename: str) -> str:
    root, ext = os.path.splitext(filename)
    path = os.path.join(base_dir, filename)
    i = 2
    while os.path.exists(path):
        path = os.path.join(base_dir, f"{root}_{i}{ext}")
        i += 1
    return path

def guess_ext_from_url(u: str) -> str:
    ext = os.path.splitext(urlparse(u or "").path)[1].lower()
    if ext:
        return ext
    u = (u or "").lower()
    if "png" in u:  return ".png"
    if "gif" in u:  return ".gif"
    if "webp" in u: return ".webp"
    if "svg" in u:  return ".svg"
    return ".jpg"

def download_to_file(url: str, dst_path: str, timeout: int = 60):
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(dst_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

total_attempts = 0
print("\n--- Downloading images from images_with_slugs ---")

for doc in images_with_slugs:
    doc_name = safe_name(doc.get("doc_name") or doc.get("file_id") or "document")
    out_dir = os.path.join(RAW_DIR, doc_name)
    os.makedirs(out_dir, exist_ok=True)

    imgs = doc.get("images", []) or []
    if not imgs:
        continue

    print(f"\n{doc_name}: {len(imgs)} image(s)")

    for i, attrs in enumerate(imgs, start=1):
        src = attrs.get("src")
        if not src:
            continue

        # prefer short slug; fallback to full; fallback to generic
        base = (attrs.get("slug_alt_first5") or attrs.get("slug_alt_full") or f"image-{i}").strip("-") or f"image-{i}"
        ext  = guess_ext_from_url(src)
        filename = f"{base}{ext}"
        dst = ensure_unique_path(out_dir, filename)

        total_attempts += 1
        try:
            print(f"  ↓ {src}\n    → {dst}")
            download_to_file(src, dst, timeout=60)
            downloaded_image_paths[src] = dst
        except Exception as e:
            download_errors.append({"src": src, "filename": filename, "doc_name": doc_name, "error": str(e)})
            print(f"    ✗ Failed: {e}")

print("\n--- Download summary ---")
print(f"Total attempted: {total_attempts}")
print(f"Successfully downloaded: {len(downloaded_image_paths)}")
print(f"Failed: {len(download_errors)}")



--- Downloading images from images_with_slugs ---

Bài viết _ 36 - kệ trung tải 4 tầng: 6 image(s)
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXeg3YVeOKMv6LSZs-Sa5mGxNBN_NVvhE1HsYVQWJJi_nIRNu9iMlR02QhZIcWCKRs3BrpkWK1Xj2NGJu5hc-1yh0euLol4cD8Qaiz_L2g5eTCrqDbJHbC27gJzTSR_wiDWcTPObwA?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/ke-trung-tai-4-tang_7.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXciuvN7msHn5wvUbaximAEko84LVKr2QXQaxUrVh7zIBoMDmGwXBm2i0Q42lT_MhEhVlwYdm_w0BWWB3iCKCy3UdgTz7ZxWXJTJh3SdkFGrl2EFgv3SfSf9PRRvEYW9ob5D4sX0BQ?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/ban-ve-chi-tiet-ke_7.jpg
  ↓ https://lh7-rt.googleusercontent.com/docsz/AD_4nXfpyaAlpERweeatN4TflG45lNxEH2Os70Xt1fOokBYS4yiUcwJMQam7fHv-AJjFsA0PiiYRNoG0aeCpI8yoLPR1FdCv02M5cDW0pqrX9hEhtb1FDy609xIvTJtX6Ukh93yfLbNT?key=5axpOQhltCUHo8WHcYAGCg
    → images_raw/Bài viết _ 36 - kệ trung tải 4 tầng/thiet-ke-ke-4-tang_7.jpg
  ↓ https://lh7-

In [135]:
## Step 10: Resize the images
# Define target width and height
target_image_width = 800  # Example width in pixels
target_image_height = 600 # Example height in pixels

try:
    downloaded_image_paths
except NameError:
    raise RuntimeError("`downloaded_image_paths` not found. Run the download step first.")

RESIZED_DIR = "images_resized"
os.makedirs(RESIZED_DIR, exist_ok=True)

resized_image_paths = {}   # {src_url: resized_local_path}
resize_errors = []

def resize_fit(src_path: str, dst_path: str, max_w: int, max_h: int):
    with Image.open(src_path) as im:
        w, h = im.size
        # no upscaling
        scale = min(max_w / w, max_h / h, 1.0)
        new_size = (max(1, int(w * scale)), max(1, int(h * scale)))

        # convert + resize
        im = im.convert("RGBA").resize(new_size, Image.LANCZOS)

        # choose save format based on extension
        ext = os.path.splitext(dst_path)[1].lower()
        if ext in {".jpg", ".jpeg"}:
            # flatten alpha for JPEG
            bg = Image.new("RGB", im.size, (255, 255, 255))
            bg.paste(im, mask=im.split()[-1])
            bg.save(dst_path, format="JPEG", quality=92, optimize=True)
        elif ext == ".png":
            im.save(dst_path, format="PNG", optimize=True)
        elif ext == ".webp":
            im.save(dst_path, format="WEBP", quality=90, method=6)
        else:
            im.save(dst_path)  # fallback: keep detected format

for src_url, local_path in downloaded_image_paths.items():
    try:
        out_path = os.path.join(RESIZED_DIR, os.path.basename(local_path))
        resize_fit(local_path, out_path, target_image_width, target_image_height)
        resized_image_paths[src_url] = out_path
        print(f"Resized → {out_path}")
    except (UnidentifiedImageError, OSError) as e:
        resize_errors.append({"source_url": src_url, "path": local_path, "error": str(e)})
        print(f"Failed to resize {local_path}: {e}")

print(f"\nResize summary: resized={len(resized_image_paths)} failed={len(resize_errors)}")


Resized → images_resized/ke-trung-tai-4-tang_7.jpg
Resized → images_resized/ban-ve-chi-tiet-ke_7.jpg
Resized → images_resized/thiet-ke-ke-4-tang_7.jpg
Resized → images_resized/uu-iem-ke-4-tang_7.jpg
Resized → images_resized/ung-dung-ke-4-tang_7.jpg
Resized → images_resized/ke-4-tang-rackcosmo_7.jpg

Resize summary: resized=6 failed=0


In [136]:
## Step 11: Setup Wordpress API

# --- CONFIG: set these once (better: use environment variables) ---
WP_BASE_URL = "https://ngoncareer.com"          # no trailing slash
WP_USERNAME  = "admin_career"
WP_APP_PASS  = "efkV mie6 u3P8 C3mC NyM8 Ickp"   # app password (as shown in WP UI)

# WordPress application passwords are shown with spaces; remove them for Basic Auth
wp_pass_clean = (WP_APP_PASS or "").replace(" ", "")

if not WP_BASE_URL or not WP_USERNAME or not wp_pass_clean:
    raise ValueError("Missing WP_BASE_URL / WP_USERNAME / WP_APP_PASS")

# Build the REST base and commonly used endpoints
WP_API_BASE = f"{WP_BASE_URL}/wp-json/wp/v2"
WP_MEDIA_EP = f"{WP_API_BASE}/media"

# Build Basic Auth header
token = base64.b64encode(f"{WP_USERNAME}:{wp_pass_clean}".encode("utf-8")).decode("utf-8")
auth_header = {"Authorization": f"Basic {token}"}

print("WordPress REST API auth header ready.")
print("Base:", WP_API_BASE)

WordPress REST API auth header ready.
Base: https://ngoncareer.com/wp-json/wp/v2


In [137]:
#Step 12: Upload images to wordpress
try:
    WP_MEDIA_EP, auth_header, resized_image_paths
except NameError:
    raise RuntimeError("Missing WP_MEDIA_EP/auth_header or resized_image_paths.")

def guess_mime(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    return {
        ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".png": "image/png", ".gif": "image/gif",
        ".webp": "image/webp", ".svg": "image/svg+xml",
    }.get(ext, "application/octet-stream")

def upload_resized(local_path: str) -> dict:
    """Upload a single *resized* image file to WP and return media JSON."""
    filename = os.path.basename(local_path)
    with open(local_path, "rb") as f:
        data = f.read()
    headers = {
        **auth_header,
        "Content-Disposition": f'attachment; filename="{filename}"',
        "Content-Type": guess_mime(local_path),
    }
    r = requests.post(WP_MEDIA_EP, headers=headers, data=data, timeout=90)
    if r.status_code >= 400:
        raise RuntimeError(f"Upload failed {r.status_code}: {r.text[:400]}")
    return r.json()

# Results:
# - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <public_url>, "name": <title>}}
# - wp_upload_errors: [{source_url, local_path, error}]
wp_uploaded_map = {}
wp_upload_errors = []

print("\n— Uploading RESIZED images to WordPress —")
for orig_src, local_path in resized_image_paths.items():
    try:
        media = upload_resized(local_path)
        wp_uploaded_map[orig_src] = {
            "id": media.get("id"),
            "wp_url": media.get("source_url"),
            "name": (media.get("title") or {}).get("rendered") or media.get("slug") or os.path.basename(local_path),
            "mime_type": media.get("mime_type"),
            "local_path": local_path,
        }
        print(f"✓ {os.path.basename(local_path)} → {wp_uploaded_map[orig_src]['wp_url']}")
    except Exception as e:
        wp_upload_errors.append({"source_url": orig_src, "local_path": local_path, "error": str(e)})
        print(f"✗ Failed: {local_path} | {e}")

print(f"\nUpload summary: uploaded={len(wp_uploaded_map)} | failed={len(wp_upload_errors)}")
# Now you can use `wp_uploaded_map[original_src]["wp_url"]` to update images_with_slugs later.



— Uploading RESIZED images to WordPress —
✓ ke-trung-tai-4-tang_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tang_7.jpg
✓ ban-ve-chi-tiet-ke_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/ban-ve-chi-tiet-ke_7.jpg
✓ thiet-ke-ke-4-tang_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/thiet-ke-ke-4-tang_7.jpg
✓ uu-iem-ke-4-tang_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/uu-iem-ke-4-tang_7.jpg
✓ ung-dung-ke-4-tang_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/ung-dung-ke-4-tang_7.jpg
✓ ke-4-tang-rackcosmo_7.jpg → https://ngoncareer.com/wp-content/uploads/2025/09/ke-4-tang-rackcosmo_7.jpg

Upload summary: uploaded=6 | failed=0


In [138]:
## Step 13: Update images_with_slugs variable
## Step: Clean + update images_with_slugs
# - Ensure no pre-existing `width` / `height` on each image, then add fresh values
# - For `new_src`: if it exists, remove it first; then set it to the new WordPress URL

import os

# Pillow for reading actual resized dimensions
try:
    from PIL import Image
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow"])
    from PIL import Image

# Requires:
# - images_with_slugs: [{doc_name, url, file_id, images:[{src, ...}]}]
# - resized_image_paths: {original_src_url: "/path/to/resized.ext"}
# - wp_uploaded_map: {original_src_url: {"id":..., "wp_url":..., "local_path":...}}

try:
    images_with_slugs, resized_image_paths, wp_uploaded_map
except NameError:
    raise RuntimeError("Missing images_with_slugs, resized_image_paths, or wp_uploaded_map.")

updated = 0
skipped = 0
errors = []

for doc in images_with_slugs:
    for img in doc.get("images", []):
        orig_src = img.get("src")
        if not orig_src:
            skipped += 1
            continue

        up = wp_uploaded_map.get(orig_src)
        local_resized = resized_image_paths.get(orig_src)

        if not up or not local_resized or not os.path.isfile(local_resized):
            # No upload info or no resized file → cannot update
            skipped += 1
            continue

        # --- 1) remove any existing width/height first
        img.pop("width", None)
        img.pop("height", None)

        # --- 2) remove existing new_src (if present), then add new one
        img.pop("new_src", None)
        img["new_src"] = up.get("wp_url")

        # --- 3) compute actual resized dimensions and add width/height
        try:
            with Image.open(local_resized) as im:
                w, h = im.size
            img["width"] = int(w)
            img["height"] = int(h)
            # optional helpful fields:
            img["uploaded_media_id"] = up.get("id")
            img["resized_local_path"] = local_resized
            updated += 1
        except Exception as e:
            errors.append({"src": orig_src, "path": local_resized, "error": str(e)})

print(f"Cleaned & updated {updated} image(s). Skipped: {skipped}. Errors: {len(errors)}")
# Optional quick peek:
# [ (img.get('src'), img.get('new_src'), img.get('width'), img.get('height'))
#   for d in images_with_slugs for img in d['images'][:3] ]
images_with_slugs

Cleaned & updated 6 image(s). Skipped: 0. Errors: 0


[{'doc_name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXeg3YVeOKMv6LSZs-Sa5mGxNBN_NVvhE1HsYVQWJJi_nIRNu9iMlR02QhZIcWCKRs3BrpkWK1Xj2NGJu5hc-1yh0euLol4cD8Qaiz_L2g5eTCrqDbJHbC27gJzTSR_wiDWcTPObwA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'slug_alt_full': 'ke-trung-tai-4-tang',
    'slug_alt_first5': 'ke-trung-tai-4-tang',
    'new_src': 'https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tang_7.jpg',
    'width': 800,
    'height': 600,
    'uploaded_media_id': 49321,
    'resized_local_path': 'images_resized/ke-tru

In [139]:
## Step 14: Extract the main heading from each exported HTML
## Step: Extract ONLY the first <h1> from each exported HTML

try:
    exported_html_docs  # [{url, file_id, name, html}]
except NameError:
    raise RuntimeError("`exported_html_docs` not found. Run the export step first.")

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

h1_summary = []  # [{file_id, name, h1}]

for doc in exported_html_docs:
    html = doc.get("html", "") or ""
    soup = BeautifulSoup(html, "html.parser")

    # only the FIRST <h1>
    h1_tag = soup.find("h1")
    h1_text = clean_text(h1_tag.get_text()) if h1_tag and h1_tag.get_text(strip=True) else None

    # save back to the doc
    doc["h1"] = h1_text

    h1_summary.append({
        "file_id": doc.get("file_id"),
        "name": doc.get("name"),
        "h1": h1_text,
    })

    if h1_text is None:
        print(f"(no <h1>) {doc.get('name') or doc.get('file_id')}")

print(f"Extracted first <h1> for {len(h1_summary)} document(s).")
# Optional peek:
# for row in h1_summary[:5]:
print(h1_summary[0]["h1"])


Extracted first <h1> for 1 document(s).
Kệ trung tải 4 tầng: Tối ưu hóa không gian theo chiều cao, tăng gấp đôi hiệu suất


In [140]:
##Step 15: Create empty post in wordpress
# Requires:
# - WP_API_BASE and auth_header from your Step 11
# - h1_summary (list of dicts with key "h1")
# - main_keyword_slug (string)

try:
    WP_API_BASE, auth_header, h1_summary, main_keyword_slug_to_run
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header or h1_summary/main_keyword_slug_to_run.")

posts_endpoint = f"{WP_API_BASE}/posts"
title = (h1_summary[0].get("h1") or "").strip()

post_data = {
    "title": title,
    "slug": main_keyword_slug_to_run[0],
    "status": "draft",
    "content": "",
}

print(f"Creating draft post at: {posts_endpoint}")
resp = requests.post(posts_endpoint, headers=auth_header, json=post_data)

if resp.status_code in (200, 201):
    data = resp.json()
    wordpress_post_id = data.get("id")
    wordpress_post_link = data.get("link")
    print(f"✓ Draft created. ID: {wordpress_post_id} | Link: {wordpress_post_link}")
else:
    raise RuntimeError(f"Post create failed {resp.status_code}: {resp.text[:500]}")


Creating draft post at: https://ngoncareer.com/wp-json/wp/v2/posts
✓ Draft created. ID: 49327 | Link: https://ngoncareer.com/?p=49327


In [141]:
##Step 16: Update images metadata
# Requires:
#   - WP_API_BASE or WP_MEDIA_EP, and auth_header (from Step 11)
#   - wordpress_post_id (created post ID)
#   - wp_uploaded_map: {original_src_url: {"id": <media_id>, "wp_url": <media_url>, ...}}
# Optional:
#   - images_with_slugs to supply alt text per original src

# ---- inputs check ----
try:
    wordpress_post_id
    auth_header
    WP_API_BASE
except NameError:
    raise RuntimeError("Missing wordpress_post_id, auth_header, or WP_API_BASE (from Step 11).")

WP_MEDIA_EP = f"{WP_API_BASE}/media"

try:
    wp_uploaded_map
except NameError:
    raise RuntimeError("Missing `wp_uploaded_map` (built after uploads).")

# Build src_url -> alt text map from images_with_slugs (optional)
src_to_alt = {}
if "images_with_slugs" in globals() and isinstance(images_with_slugs, list):
    for doc in images_with_slugs:
        for img in doc.get("images", []):
            src = img.get("src")
            if not src:
                continue
            alt = (img.get("alt") or "").strip()
            # first win
            src_to_alt.setdefault(src, alt)

updated, failed = 0, []
print("\n— Updating media metadata and associating with the post —")

for original_src, meta in wp_uploaded_map.items():
    media_id = meta.get("id")
    if not media_id:
        failed.append({"src": original_src, "error": "Missing media_id"})
        continue

    alt = src_to_alt.get(original_src, "").strip()

    payload = {
        "alt_text": alt,                 # alt on image
        "caption": alt,                  # caption text
        "description": alt,              # description text
        "post": int(wordpress_post_id),  # attach to the created post
    }

    try:
        r = requests.post(
            f"{WP_MEDIA_EP}/{media_id}",
            headers={**auth_header, "Content-Type": "application/json"},
            json=payload,
            timeout=60,
        )
        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
        updated += 1
        print(f"✓ Media {media_id} updated")
    except Exception as e:
        failed.append({"media_id": media_id, "src": original_src, "error": str(e)})
        print(f"✗ Media {media_id} failed: {e}")

print(f"\nSummary: updated={updated} | failed={len(failed)}")
# Optional peek:
# failed[:3]

# Append the WordPress post_id onto every image entry in images_with_slugs
## Append/refresh WordPress post_id on images_with_slugs (no duplicates)

try:
    wordpress_post_id
    images_with_slugs
except NameError:
    raise RuntimeError("Missing `wordpress_post_id` or `images_with_slugs`.")

pid = int(wordpress_post_id)
set_count = 0
already_count = 0
corrected_count = 0

for doc in images_with_slugs:
    for img in doc.get("images", []):
        if "post_id" in img:
            # If it's different (or wrong type), refresh it; otherwise leave as-is
            if img["post_id"] != pid:
                img["post_id"] = pid
                corrected_count += 1
            else:
                already_count += 1
        else:
            img["post_id"] = pid
            set_count += 1

print(
    f"post_id updates → set:{set_count}, corrected:{corrected_count}, already-correct:{already_count}"
)
images_with_slugs


— Updating media metadata and associating with the post —
✓ Media 49321 updated
✓ Media 49322 updated
✓ Media 49323 updated
✓ Media 49324 updated
✓ Media 49325 updated
✓ Media 49326 updated

Summary: updated=6 | failed=0
post_id updates → set:6, corrected:0, already-correct:0


[{'doc_name': 'Bài viết | 36 - kệ trung tải 4 tầng',
  'url': 'https://docs.google.com/document/d/19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E/edit?usp=sharing',
  'file_id': '19gHsZgK9TSTXaQ65sM-oO_7kDIKIwNL0tLIixyYqu0E',
  'images': [{'alt': 'Kệ trung tải 4 tầng',
    'src': 'https://lh7-rt.googleusercontent.com/docsz/AD_4nXeg3YVeOKMv6LSZs-Sa5mGxNBN_NVvhE1HsYVQWJJi_nIRNu9iMlR02QhZIcWCKRs3BrpkWK1Xj2NGJu5hc-1yh0euLol4cD8Qaiz_L2g5eTCrqDbJHbC27gJzTSR_wiDWcTPObwA?key=5axpOQhltCUHo8WHcYAGCg',
    'style': 'width: 601.70px; height: 450.67px; margin-left: 0.00px; margin-top: 0.00px; transform: rotate(0.00rad) translateZ(0px); -webkit-transform: rotate(0.00rad) translateZ(0px);',
    'title': '',
    'slug_alt_full': 'ke-trung-tai-4-tang',
    'slug_alt_first5': 'ke-trung-tai-4-tang',
    'new_src': 'https://ngoncareer.com/wp-content/uploads/2025/09/ke-trung-tai-4-tang_7.jpg',
    'width': 800,
    'height': 600,
    'uploaded_media_id': 49321,
    'resized_local_path': 'images_resized/ke-tru

In [142]:
# Step 17 (VS Code): Replace <img> tags with WordPress [caption] shortcodes (src + alt + width + height)
# Strategy:
#   1) Try exact per-doc match by original `src` from images_with_slugs.
#   2) If no exact match (Docs regenerates URLs), fall back to order: 1st <img> ↔ 1st image data, etc.

from bs4 import BeautifulSoup, NavigableString
from html import unescape as html_unescape  # avoid shadowing the html module

# Requires:
# - exported_html_docs: [{file_id, name, html, ...}]
# - images_with_slugs : [{file_id or doc_name/name/url, images:[{src, new_src/new_url, alt, width/new_width, height/new_height, uploaded_media_id}, ...]}]

try:
    exported_html_docs, images_with_slugs
except NameError:
    raise RuntimeError("Missing `exported_html_docs` or `images_with_slugs`. Run previous steps first.")

DEFAULT_ALIGN = "aligncenter"
DEFAULT_SIZE_CLASS = "size-full"

# Group images_with_slugs by file_id (preferred) and by name (fallback)
by_file_id, by_name = {}, {}
for d in images_with_slugs:
    key_id = d.get("file_id")
    key_name = d.get("doc_name") or d.get("name") or d.get("url")
    images = d.get("images", []) or []
    if key_id:
        by_file_id.setdefault(key_id, []).extend(images)
    if key_name:
        by_name.setdefault(key_name, []).extend(images)

def build_caption_shortcode(info: dict) -> str:
    """
    Build a WP [caption]...[/caption] shortcode string containing a full <img .../> with:
      - class="aligncenter size-full wp-image-{id}" (if id available)
      - title="" (empty, to match your pattern)
      - src, alt, width, height (when present)
    """
    new_src   = info.get("new_src") or info.get("new_url")
    new_alt   = (info.get("alt") or "").strip()
    new_width = info.get("width") or info.get("new_width")
    new_height= info.get("height") or info.get("new_height")
    media_id  = info.get("uploaded_media_id")

    classes = f'{DEFAULT_ALIGN} {DEFAULT_SIZE_CLASS}'
    if media_id:
        classes += f' wp-image-{media_id}'

    bits = [f'<img class="{classes}" title="" src="{new_src}"']
    if new_alt:
        bits.append(f' alt="{new_alt}"')
    if new_width is not None:
        try: bits.append(f' width="{int(new_width)}"')
        except Exception: bits.append(f' width="{new_width}"')
    if new_height is not None:
        try: bits.append(f' height="{int(new_height)}"')
        except Exception: bits.append(f' height="{new_height}"')
    bits.append(" />")
    inner_img = "".join(bits)

    cap_width = ""
    if new_width is not None:
        try: cap_width = str(int(new_width))
        except Exception: cap_width = str(new_width)

    return f'[caption id="attachment_{media_id or ""}" align="{DEFAULT_ALIGN}" width="{cap_width}"]{inner_img} {new_alt}[/caption]'

processed_html_docs = []   # [{file_id, name, html_processed, images_replaced}]
total_replaced = 0

for doc in exported_html_docs:
    file_id = doc.get("file_id")
    name    = doc.get("name")
    html_in = doc.get("html") or ""

    soup = BeautifulSoup(html_in, "html.parser")
    img_tags = soup.find_all("img")

    # per-doc image data
    imgs_list = by_file_id.get(file_id)
    if imgs_list is None:
        imgs_list = by_name.get(name, [])
    per_doc_src_map = { img.get("src"): img for img in imgs_list if img.get("src") }

    n_html = len(img_tags)
    n_data = len(imgs_list)
    n_replace = min(n_html, n_data)  # fallback upper-bound when no exact match
    replaced = 0
    idx_fallback = 0

    for i, tag in enumerate(img_tags):
        orig_src = tag.get("src")
        matched_info = None

        # 1) exact per-doc src match
        if orig_src in per_doc_src_map:
            matched_info = per_doc_src_map[orig_src]

        # 2) fallback by position if still no match
        if matched_info is None and idx_fallback < n_data:
            matched_info = imgs_list[idx_fallback]
            idx_fallback += 1

        if not matched_info:
            continue
        if not (matched_info.get("new_src") or matched_info.get("new_url")):
            continue

        shortcode = build_caption_shortcode(matched_info)
        tag.replace_with(NavigableString(shortcode))
        replaced += 1

    processed_html_docs.append({
        "file_id": file_id,
        "name": name,
        "html_processed": html_unescape(str(soup)),  # ensure shortcodes not HTML-escaped
        "images_replaced": replaced,
    })
    total_replaced += replaced

print(f"Processed {len(processed_html_docs)} document(s). Total images replaced: {total_replaced}")

# Optional: handy dict for further steps (e.g., updating WP post content)
processed_html_contents = { (d["file_id"] or d["name"]): d["html_processed"] for d in processed_html_docs }


Processed 1 document(s). Total images replaced: 6


In [143]:
# Step: Robust HTML transforms (VS Code friendly)
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urlparse, parse_qs, unquote
import re

def _parse_style(style_str: str) -> dict:
    """Turn inline style string into a dict; case-insensitive keys."""
    out = {}
    if not style_str:
        return out
    for frag in style_str.split(";"):
        if not frag.strip():
            continue
        if ":" not in frag:
            continue
        k, v = frag.split(":", 1)
        out[k.strip().lower()] = v.strip()
    return out

def _style_to_str(style_dict: dict) -> str:
    """Back to 'k: v; k2: v2' (preserve order not needed)."""
    if not style_dict:
        return ""
    return "; ".join(f"{k}: {v}" for k, v in style_dict.items())

def transform_html_dom(html_in: str) -> str:
    html_in = html_in or ""
    soup = BeautifulSoup(html_in, "html.parser")

    changed = {
        "img_style_removed": 0,
        "cmnt_blocks_removed": 0,
        "h1_removed": 0,
        "p_normalized":0,
        "p_img_centered_wrapped":0,
        "p_title_removed": 0,
        "table_width_added": 0,
        "style_tags_removed": 0,
        "span_bold_to_strong":0,
        "span_unwrapped":0,
        "links_unwrapped": 0,
        "font_props_removed": 0,
        "italic_wrapped": 0,
    }
  
    # 0) Mark italic nodes BEFORE we strip styles
    for el in soup.select("[style]"):
        st = el.get("style", "")
        if "font-style" in st.lower() and "italic" in st.lower():
            el["data-italic-mark"] = "1"

    # 1) REMOVE <img style="...">
    for img in soup.find_all("img"):
        if "style" in img.attrs:
            del img.attrs["style"]
            changed["img_style_removed"] += 1

    # 2) REMOVE COMMENTS BLOCKS like: <div><p><a id="cmnt123">[...]</a><span>...</span></p></div>
    #    We'll remove the nearest wrapping <div> that contains that pattern.
    for a in soup.find_all("a", id=re.compile(r"^cmnt\d+$", re.IGNORECASE)):
        # try to match the structure div > p > a + span
        div = a.find_parent("div")
        if not div:
            continue
        p = a.find_parent("p")
        if not p or p.find_parent("div") is not div:
            continue
        span = p.find("span")
        if not span:
            continue
        div.decompose()
        changed["cmnt_blocks_removed"] += 1

    # 3) REMOVE ALL <h1>
    for h1 in soup.find_all("h1"):
        h1.decompose()
        changed["h1_removed"] += 1

    # 4) REMOVE <p class="title">...</p> (case-insensitive)
    for p in list(soup.find_all("p")):
        cls = p.get("class", [])
        if isinstance(cls, str):
            cls = [cls]
        cls_lower = {c.lower() for c in cls}
        if "title" in cls_lower:
            p.decompose()
            changed["p_title_removed"] += 1
    # 4a) Normalize ALL <p> tags to dir="ltr" + text-align (center/justify/right; default justify)
    for p in soup.find_all("p"):
        st = _parse_style(p.get("style", ""))
        align_attr = (p.get("align") or "").strip().lower()
        style_align = (st.get("text-align") or "").strip().lower()

        has_center  = (style_align == "center")  or (align_attr == "center")
        has_justify = (style_align == "justify") or (align_attr == "justify")
        has_right   = (style_align == "right")   or (align_attr == "right")

        p["dir"] = "ltr"
        new_style = {}
        if has_center:
            new_style["text-align"] = "center"
        elif has_justify:
            new_style["text-align"] = "justify"
        elif has_right:
            new_style["text-align"] = "right"
        else:
            new_style["text-align"] = "justify"

        p["style"] = _style_to_str(new_style)
        if "align" in p.attrs:
            del p.attrs["align"]
        changed["p_normalized"] += 1

    # 4b) Ensure any <p> that contains an <img> gets centered and wrapped with <span style="font-weight: 400">…</span>
    for p in soup.find_all("p"):
        if p.find("img"):
            # strip existing style, then enforce center
            if "style" in p.attrs:
                del p.attrs["style"]
            p["style"] = "text-align: center;"

            # avoid double-wrap if already exactly wrapped
            only_child = p.contents[0] if p.contents else None
            already_wrapped = (
                len(p.contents) == 1
                and getattr(only_child, "name", "") == "span"
                and (only_child.get("style") or "").replace(" ", "").rstrip(";") == "font-weight:400"
            )
            if not already_wrapped:
                wrapper = soup.new_tag("span")
                wrapper["style"] = "font-weight: 400"
                while p.contents:
                    wrapper.append(p.contents[0].extract())
                p.append(wrapper)
                changed["p_img_centered_wrapped"] += 1

    # 5) TABLE MAX WIDTH (ensure width:100% present in style)
    for table in soup.find_all("table"):
        sd = _parse_style(table.get("style", ""))
        # Only add width:100% if not already present
        has_width = any(k.lower() == "width" for k in sd)
        if not has_width:
            sd["width"] = "100%"
            table["style"] = _style_to_str(sd)
            changed["table_width_added"] += 1
        else:
            # keep existing style but don't count as change
            table["style"] = _style_to_str(sd)

    # 6) REMOVE all <style>…</style> blocks
    for st in soup.find_all("style"):
        st.decompose()
        changed["style_tags_removed"] += 1

    # 7) REMOVE google.com redirect in hrefs
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("https://www.google.com/url?"):
            qs = parse_qs(urlparse(href).query)
            q = qs.get("q", [None])[0]
            if q:
                a["href"] = unquote(q)
                changed["links_unwrapped"] += 1

    # 8) REMOVE FONT FAMILY / SIZE and font-style: italic in inline style
    for el in soup.select("[style]"):
        sd = _parse_style(el.get("style", ""))
        before = sd.copy()
        sd.pop("font-family", None)
        sd.pop("font-size", None)
        # remove font-style entirely (the italic will be handled by wrapping below)
        if "font-style" in sd:
            del sd["font-style"]
        if sd:
            el["style"] = _style_to_str(sd)
        else:
            del el.attrs["style"]
        if sd != before:
            changed["font_props_removed"] += 1
    # 8a) CHANGE <span style="font-weight: bold|700">…</span> → <strong>…</strong>
    for span in list(soup.find_all("span")):
        st = _parse_style(span.get("style", ""))
        fw = (st.get("font-weight") or "").strip().lower()
        if fw in {"bold", "700"}:
            strong = soup.new_tag("strong")
            while span.contents:
                strong.append(span.contents[0].extract())
            span.replace_with(strong)
            changed["span_bold_to_strong"] += 1

    # 8b) Remove (unwrap) any remaining <span> tags while keeping content
    for span in list(soup.find_all("span")):
        span.unwrap()
        changed["span_unwrapped"] += 1

    # 9) FIX ITALIC: wrap contents of marked nodes in <em>…</em>
    for el in soup.find_all(attrs={"data-italic-mark": True}):
        # Avoid double wrapping: if already has a single child <em> covering all, skip
        if len(el.contents) == 1 and getattr(el.contents[0], "name", "") == "em":
            pass
        else:
            em = soup.new_tag("em")
            while el.contents:
                em.append(el.contents[0].extract())
            el.append(em)
            changed["italic_wrapped"] += 1
        del el.attrs["data-italic-mark"]

    # 10) Serialize
    out = str(soup)

    # 11) Replace &nbsp; (entity and unicode) and collapse front row leading space as per your JS
    out = out.replace("&nbsp;", " ").replace("\u00a0", " ")
    out = re.sub(r"^(\s*)", lambda m: m.group(1).replace("\u00a0", " "), out)  # safety
    out = re.sub(r"^(\s*&nbsp;\s*)+", "", out, flags=re.IGNORECASE)

    # (Your JS also collapsed ALL whitespace globally; that can break HTML formatting.
    # If you truly want it, uncomment the next two lines.)
    # out = re.sub(r"\s+", " ", out).strip()

    # Quick summary so you can verify it actually did work:
    print("Transform summary:", {k: v for k, v in changed.items() if v})

    return out


In [144]:
# Step 19: Apply changes
# Apply to whatever HTML you’re about to send to WordPress:
try:
    processed_html_docs
except NameError:
    raise RuntimeError("Run the image-replacement step first to build `processed_html_docs`.")

changed = 0
for d in processed_html_docs:
    src = d.get("html_processed") or d.get("html") or ""
    out = transform_html_dom(src)
    if out != src:
        changed += 1
    d["html_processed"] = out

print(f"Transformed {changed}/{len(processed_html_docs)} document(s).")


Transform summary: {'h1_removed': 1, 'p_normalized': 41, 'p_img_centered_wrapped': 6, 'table_width_added': 1, 'style_tags_removed': 1, 'span_bold_to_strong': 30, 'span_unwrapped': 83, 'links_unwrapped': 3, 'font_props_removed': 91}
Transformed 1/1 document(s).


In [145]:
## Step 18: Update content of post in wordpress

# Requires:
# - WP_API_BASE, auth_header (from Step 11)
# - wordpress_post_id (from Step 15)
# - processed_html_docs (from the replace step)

try:
    WP_API_BASE, auth_header, wordpress_post_id, processed_html_docs
except NameError:
    raise RuntimeError("Missing WP_API_BASE/auth_header, wordpress_post_id, or processed_html_docs.")

if not processed_html_docs:
    raise RuntimeError("processed_html_docs is empty. Run the HTML processing step first.")

# Choose which processed doc to insert (here: the first one)
html_str = processed_html_docs[0]["html_processed"]

posts_endpoint = f"{WP_API_BASE}/posts/{int(wordpress_post_id)}"
payload = {
    "content": html_str,   # raw HTML (can include shortcodes)
    # "status": "draft",   # optional: keep as draft
    # "status": "publish"  # optional: publish immediately
}

print(f"Updating post #{wordpress_post_id} at: {posts_endpoint}")
resp = requests.post(
    posts_endpoint,
    headers={**auth_header, "Content-Type": "application/json"},
    json=payload,
    timeout=90,
)

if resp.status_code == 200:
    data = resp.json()
    print(f"✓ Post updated. View: {data.get('link')}")
else:
    raise RuntimeError(f"Update failed {resp.status_code}: {resp.text[:500]}")


Updating post #49327 at: https://ngoncareer.com/wp-json/wp/v2/posts/49327
✓ Post updated. View: https://ngoncareer.com/?p=49327
